# GPT-2 + QQP — LoRA vs LoRA+

**Exactly nine 3-epoch screening runs (seed 42), then six 3-epoch final runs (seeds 42, 43, 44). No smoke-test runs.**

Start with a **T4 GPU** in Colab. Execute cells from top to bottom. Screening is enabled. After all nine finish, execute winner selection, then set `RUN_FINAL=True` in the final evaluation cell. Saves results and checkpoints to Google Drive.

The nine screening settings, evaluation every 200 steps, checkpoint every 1,000 steps, and binary F1 winner selection follow the team README. Physical batch size 8 with accumulation 4 provides effective batch size 32.

## 1. Install dependencies (Google Colab)
Run once. If prompted, restart runtime, then continue from the next cell.

In [2]:
%pip -q install "transformers==4.57.1" "peft==0.17.1" "datasets==3.6.0" "accelerate==1.10.1" "scikit-learn>=1.4,<2" "pandas>=2,<3" "matplotlib>=3.8,<4"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


## 2. Imports, Drive storage and configuration

In [5]:
import os, json, gc, time, random, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer,
                          get_linear_schedule_with_warmup, set_seed)
from peft import LoraConfig, TaskType, get_peft_model

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/LoRA_Project/Asneem_GPT2_QQP_50k')
except ImportError:
    ROOT = Path('./Asneem_GPT2_QQP_50k')
ROOT.mkdir(parents=True, exist_ok=True)
print('Output directory:', ROOT)
print('CUDA:', torch.cuda.is_available(), 'GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Use Runtime > Change runtime type > GPU before training.'

CFG = dict(model_name='gpt2', dataset='qqp', subset_seed=42, train_samples=50_000,
           max_length=128, rank=8, lora_alpha=8, lora_dropout=0.1,
           target_modules=['c_attn','c_proj','c_fc'], epochs=3,
           train_batch_size=8, eval_batch_size=32, gradient_accumulation_steps=4,
           adam_beta1=0.9, adam_beta2=0.99, adam_epsilon=1e-8,
           weight_decay=0.01, warmup_steps=0, final_seeds=[42,43,44])
# Physical batch 8 x accumulation 4 = effective batch 32 on Colab T4.
# Classification head LR=LR_A is a documented implementation choice.
(ROOT/'config.json').write_text(json.dumps(CFG,indent=2))
print(json.dumps(CFG,indent=2))

Mounted at /content/drive
Output directory: /content/drive/MyDrive/LoRA_Project/Asneem_GPT2_QQP_50k
CUDA: True GPU: Tesla T4
{
  "model_name": "gpt2",
  "dataset": "qqp",
  "subset_seed": 42,
  "train_samples": 50000,
  "max_length": 128,
  "rank": 8,
  "lora_alpha": 8,
  "lora_dropout": 0.1,
  "target_modules": [
    "c_attn",
    "c_proj",
    "c_fc"
  ],
  "epochs": 3,
  "train_batch_size": 8,
  "eval_batch_size": 32,
  "gradient_accumulation_steps": 4,
  "adam_beta1": 0.9,
  "adam_beta2": 0.99,
  "adam_epsilon": 1e-08,
  "weight_decay": 0.01,
  "warmup_steps": 0,
  "final_seeds": [
    42,
    43,
    44
  ]
}


### Reproducibility and protocol notes
- Fixed **subset seed 42** means all runs see exactly the same 50k examples; model seed changes between final runs.
- Training uses **3 epochs**, max length **128**, and effective batch size **32**. If a T4 runs out of memory, set `train_batch_size=8` and `gradient_accumulation_steps=4`; this preserves effective batch size but is not identical to physical batch 32.
- We use **A randomly initialized / B zero initialized** adapters, `score` classifier head trainable, and dynamic padding.
- GPT-2 question formatting and classifier-head optimizer policy are implementation choices; exact reproduction of the authors' repository requires a separate code audit.
- Each run starts from a fresh pretrained GPT-2 model, not the previous run's weights.

## 3. Load the fixed 50k QQP training subset and full validation split

In [6]:
raw = load_dataset('nyu-mll/glue', 'qqp')
train_raw = raw['train'].shuffle(seed=CFG['subset_seed']).select(range(CFG['train_samples']))
valid_raw = raw['validation']
assert len(train_raw)==50_000 and len(valid_raw)==40_430
# Save a fingerprint of the selected dataset so experiments can be audited.
subset_keys = [str(x) for x in train_raw['idx']]
subset_digest = hashlib.sha256(','.join(subset_keys).encode()).hexdigest()
(ROOT/'subset_sha256.txt').write_text(subset_digest+'\n')
print('Train:',len(train_raw),'Validation:',len(valid_raw),'Subset SHA256:',subset_digest)
print(train_raw[0])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

qqp/train-00000-of-00001.parquet:   0%|          | 0.00/33.6M [00:00<?, ?B/s]

qqp/validation-00000-of-00001.parquet:   0%|          | 0.00/3.73M [00:00<?, ?B/s]

qqp/test-00000-of-00001.parquet:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]

Train: 50000 Validation: 40430 Subset SHA256: 4b6d64cbab6576ed317169322bdaf13629f11daabb7697c0034ee02de37c1c01
{'question1': 'What would Rhaegar and Jon have thought of each other?', 'question2': 'What is the amount of torque a 2000 Jeep Cherokee can output?', 'label': 0, 'idx': 260796}


## 4. GPT-2 tokenization (same for every run)

In [7]:
tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
tokenizer.pad_token = tokenizer.eos_token

def tokenize_qqp(batch):
    # This pair format is a documented project choice, not confirmed authors' format.
    text = [f'Question 1: {a}\nQuestion 2: {b}'
            for a,b in zip(batch['question1'],batch['question2'])]
    return tokenizer(text, truncation=True, max_length=CFG['max_length'], padding=False)

train_ds = train_raw.map(tokenize_qqp, batched=True,
                         remove_columns=['question1','question2','idx'], desc='Tokenize train')
valid_ds = valid_raw.map(tokenize_qqp, batched=True,
                         remove_columns=['question1','question2','idx'], desc='Tokenize validation')
collator = DataCollatorWithPadding(tokenizer=tokenizer)
assert 'label' in train_ds.column_names
print('Train columns:',train_ds.column_names)
print('Sample token length:',len(train_ds[0]['input_ids']))

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenize train:   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenize validation:   0%|          | 0/40430 [00:00<?, ? examples/s]

Train columns: ['label', 'input_ids', 'attention_mask']
Sample token length: 34


## 5. Fresh model, optimizer, metrics, and experiment runner

In [9]:
def fresh_model(seed):
    set_seed(seed)
    base = AutoModelForSequenceClassification.from_pretrained(
        CFG['model_name'], num_labels=2, pad_token_id=tokenizer.pad_token_id)
    adapter = LoraConfig(task_type=TaskType.SEQ_CLS,
        r=CFG['rank'], lora_alpha=CFG['lora_alpha'], lora_dropout=CFG['lora_dropout'],
        target_modules=CFG['target_modules'], fan_in_fan_out=True,
        modules_to_save=['score'], bias='none', init_lora_weights=True)
    model = get_peft_model(base, adapter)
    return model

def make_optimizer(model, lr_A, lr_B):
    groups = {'A':[], 'B':[], 'head':[]}
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        if 'lora_A' in name: groups['A'].append(p)
        elif 'lora_B' in name: groups['B'].append(p)
        else: groups['head'].append(p)
    assert groups['A'] and groups['B'] and groups['head'], 'Missing trainable parameter group'
    return torch.optim.AdamW([
        {'params':groups['A'],'lr':lr_A},
        {'params':groups['B'],'lr':lr_B},
        {'params':groups['head'],'lr':lr_A},
    ], betas=(CFG['adam_beta1'],CFG['adam_beta2']),
       eps=CFG['adam_epsilon'], weight_decay=CFG['weight_decay'])

def metrics_fn(pred):
    logits, labels = pred
    predicted = np.argmax(logits,axis=-1)
    return {'accuracy':float(accuracy_score(labels,predicted)),
            'f1':float(f1_score(labels,predicted,zero_division=0)),
            'macro_f1':float(f1_score(labels,predicted,average='macro',zero_division=0))}

def run_experiment(lr_A, ratio, seed, stage, train_data=None, valid_data=None,
                   epochs=None, save_adapter=False):
    lr_B = lr_A * ratio
    train_data = train_ds if train_data is None else train_data
    valid_data = valid_ds if valid_data is None else valid_data
    epochs = CFG['epochs'] if epochs is None else epochs
    run_id = f'{stage}_A{lr_A:.0e}_ratio{ratio}_seed{seed}'
    out = ROOT / stage / run_id
    out.mkdir(parents=True,exist_ok=True)
    result_file = out/'result.json'
    if result_file.exists():
        print('SKIPPING completed run:',run_id)
        return json.loads(result_file.read_text())
    model = fresh_model(seed)
    optimizer = make_optimizer(model,lr_A,lr_B)
    total_steps = int(np.ceil(len(train_data)/(CFG['train_batch_size'] * CFG['gradient_accumulation_steps'])) * epochs)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
    args = TrainingArguments(
        output_dir=str(out), num_train_epochs=epochs,
        per_device_train_batch_size=CFG['train_batch_size'],
        per_device_eval_batch_size=CFG['eval_batch_size'],
        gradient_accumulation_steps=CFG['gradient_accumulation_steps'],
        eval_strategy='steps', eval_steps=200,
        save_strategy='steps',save_steps=1000,save_total_limit=2,
        logging_strategy='steps',logging_steps=100,
        report_to='none',fp16=torch.cuda.is_available(),
        seed=seed,data_seed=seed, remove_unused_columns=True,
        dataloader_num_workers=0)
    trainer = Trainer(model=model,args=args,train_dataset=train_data,
                      eval_dataset=valid_data,processing_class=tokenizer,
                      data_collator=collator,compute_metrics=metrics_fn,
                      optimizers=(optimizer,scheduler))
    torch.cuda.reset_peak_memory_stats()
    started=time.perf_counter()
    from transformers.trainer_utils import get_last_checkpoint
    last_checkpoint = get_last_checkpoint(str(out))
    trained=trainer.train(resume_from_checkpoint=last_checkpoint) if last_checkpoint else trainer.train()
    measured=trainer.evaluate()
    elapsed=time.perf_counter()-started
    result = dict(stage=stage,run_id=run_id,seed=int(seed),lr_A=float(lr_A),
        lr_B=float(lr_B),ratio=int(ratio),method='LoRA' if ratio==1 else 'LoRA+',
        train_samples=len(train_data),validation_samples=len(valid_data),epochs=float(epochs),
        accuracy=float(measured['eval_accuracy']),f1=float(measured['eval_f1']),
        macro_f1=float(measured['eval_macro_f1']),
        eval_loss=float(measured['eval_loss']),train_runtime_seconds=float(trained.metrics['train_runtime']),
        total_runtime_seconds=float(elapsed),
        peak_gpu_memory_gb=float(torch.cuda.max_memory_allocated()/1024**3),
        trainable_parameters=int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
        global_steps=int(trainer.state.global_step),subset_sha256=subset_digest)
    # Store convergence history even if the Colab session later disconnects.
    (out/'trainer_log_history.json').write_text(json.dumps(trainer.state.log_history,indent=2))
    if save_adapter:
        trainer.save_model(str(out/'adapter'))
        tokenizer.save_pretrained(str(out/'adapter'))
    result_file.write_text(json.dumps(result,indent=2))
    print(json.dumps(result,indent=2))
    del trainer, model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()
    return result

## 7. Screening — nine independent cells

Run each cell separately, in order or at your convenience. Each run trains for **3 epochs** on the **same 50,000 QQP examples** using **seed 42**. Each saves its own checkpoints and results to Drive. Previously completed runs are skipped automatically. **Do not use Run all** unless you want all nine to execute sequentially.


In [10]:
SCREENING = [
    {'lr_A':0.0001, 'ratio':1},
    {'lr_A':0.0002, 'ratio':1},
    {'lr_A':0.0004, 'ratio':1},
    {'lr_A':0.0001, 'ratio':4},
    {'lr_A':0.0001, 'ratio':8},
    {'lr_A':0.0001, 'ratio':16},
    {'lr_A':0.0002, 'ratio':4},
    {'lr_A':0.0002, 'ratio':8},
    {'lr_A':0.0002, 'ratio':16},
]
assert len(SCREENING) == 9
print("Nine screening settings ready. Run the individual experiment cells below.")


Nine screening settings ready. Run the individual experiment cells below.


### Screening 1/9 — LoRA | λ = 1

LR A = `0.0001` · LR B = `0.0001` · 3 epochs · seed 42


In [7]:
result_screen_1 = run_experiment(lr_A=0.0001, ratio=1, seed=42, stage='screening')


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,Accuracy,F1,Macro F1
200,0.675800,0.603404,0.686198,0.603915,0.672045
400,0.570400,0.506904,0.735098,0.615467,0.706711


Step,Training Loss,Validation Loss,Accuracy,F1,Macro F1
200,0.675800,0.603404,0.686198,0.603915,0.672045
400,0.570400,0.506904,0.735098,0.615467,0.706711
600,0.520300,0.472408,0.756814,0.687754,0.744306
800,0.496300,0.455062,0.770245,0.707313,0.759108
1000,0.478400,0.443801,0.779050,0.727012,0.770719
1200,0.459000,0.436413,0.786545,0.705239,0.768967
1400,0.462500,0.423931,0.791145,0.734916,0.781305
1600,0.453500,0.421619,0.794657,0.712036,0.776237
1800,0.450500,0.420550,0.794781,0.750968,0.788226
2000,0.427500,0.419769,0.798046,0.719464,0.780851


{
  "stage": "screening",
  "run_id": "screening_A1e-04_ratio1_seed42",
  "seed": 42,
  "lr_A": 0.0001,
  "lr_B": 0.0001,
  "ratio": 1,
  "method": "LoRA",
  "train_samples": 50000,
  "validation_samples": 40430,
  "epochs": 3.0,
  "accuracy": 0.8161513727430126,
  "f1": 0.7623493301787255,
  "macro_f1": 0.8062195393406182,
  "eval_loss": 0.3871118724346161,
  "train_runtime_seconds": 4035.0566,
  "total_runtime_seconds": 4137.260919068,
  "peak_gpu_memory_gb": 1.5954842567443848,
  "trainable_parameters": 1181184,
  "global_steps": 4689,
  "subset_sha256": "4b6d64cbab6576ed317169322bdaf13629f11daabb7697c0034ee02de37c1c01"
}


### Screening 2/9 — LoRA | λ = 1

LR A = `0.0002` · LR B = `0.0002` · 3 epochs · seed 42


In [8]:
result_screen_2 = run_experiment(lr_A=0.0002, ratio=1, seed=42, stage='screening')


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,Accuracy,F1,Macro F1
200,0.591600,0.521444,0.720628,0.665363,0.712795
400,0.518100,0.464734,0.765941,0.708283,0.756426
600,0.478200,0.445129,0.779248,0.728385,0.771226
800,0.461500,0.425089,0.792580,0.723289,0.778704
1000,0.453400,0.419353,0.795770,0.751707,0.789129
1200,0.431400,0.410813,0.802102,0.723465,0.784691


Step,Training Loss,Validation Loss,Accuracy,F1,Macro F1
200,0.591600,0.521444,0.720628,0.665363,0.712795
400,0.518100,0.464734,0.765941,0.708283,0.756426
600,0.478200,0.445129,0.779248,0.728385,0.771226
800,0.461500,0.425089,0.792580,0.723289,0.778704
1000,0.453400,0.419353,0.795770,0.751707,0.789129
1200,0.431400,0.410813,0.802102,0.723465,0.784691
1400,0.438200,0.395619,0.810537,0.760894,0.802002
1600,0.423400,0.392660,0.812985,0.747065,0.799357
1800,0.414200,0.411135,0.800618,0.766666,0.796305
2000,0.405600,0.415007,0.803463,0.710127,0.780730


{
  "stage": "screening",
  "run_id": "screening_A2e-04_ratio1_seed42",
  "seed": 42,
  "lr_A": 0.0002,
  "lr_B": 0.0002,
  "ratio": 1,
  "method": "LoRA",
  "train_samples": 50000,
  "validation_samples": 40430,
  "epochs": 3.0,
  "accuracy": 0.8300024734108336,
  "f1": 0.7763059397884459,
  "macro_f1": 0.8196080412016928,
  "eval_loss": 0.36700746417045593,
  "train_runtime_seconds": 3983.8112,
  "total_runtime_seconds": 4086.193790116,
  "peak_gpu_memory_gb": 1.5978035926818848,
  "trainable_parameters": 1181184,
  "global_steps": 4689,
  "subset_sha256": "4b6d64cbab6576ed317169322bdaf13629f11daabb7697c0034ee02de37c1c01"
}


### Screening 3/9 — LoRA | λ = 1

LR A = `0.0004` · LR B = `0.0004` · 3 epochs · seed 42


In [9]:
result_screen_3 = run_experiment(lr_A=0.0004, ratio=1, seed=42, stage='screening')


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,Accuracy,F1,Macro F1
200,0.534900,0.575030,0.714791,0.699753,0.714074
400,0.491900,0.479760,0.759090,0.733428,0.756836
600,0.451700,0.431689,0.790477,0.748598,0.784497


Step,Training Loss,Validation Loss,Accuracy,F1,Macro F1
200,0.534900,0.575030,0.714791,0.699753,0.714074
400,0.491900,0.479760,0.759090,0.733428,0.756836
600,0.451700,0.431689,0.790477,0.748598,0.784497
800,0.441600,0.401886,0.806258,0.754459,0.797234
1000,0.433300,0.415067,0.798590,0.761810,0.793670
1200,0.413700,0.387661,0.812144,0.729860,0.792932
1400,0.411900,0.375181,0.822236,0.770669,0.812769
1600,0.401700,0.382914,0.823522,0.759497,0.810061
1800,0.387500,0.402734,0.810463,0.778379,0.806405
2000,0.393100,0.375800,0.822434,0.755991,0.808214


{
  "stage": "screening",
  "run_id": "screening_A4e-04_ratio1_seed42",
  "seed": 42,
  "lr_A": 0.0004,
  "lr_B": 0.0004,
  "ratio": 1,
  "method": "LoRA",
  "train_samples": 50000,
  "validation_samples": 40430,
  "epochs": 3.0,
  "accuracy": 0.8410338857284195,
  "f1": 0.7907945704892418,
  "macro_f1": 0.8313054605173626,
  "eval_loss": 0.3549715280532837,
  "train_runtime_seconds": 3995.4498,
  "total_runtime_seconds": 4098.286161485001,
  "peak_gpu_memory_gb": 1.5978035926818848,
  "trainable_parameters": 1181184,
  "global_steps": 4689,
  "subset_sha256": "4b6d64cbab6576ed317169322bdaf13629f11daabb7697c0034ee02de37c1c01"
}


### Screening 4/9 — LoRA+ | λ = 4

LR A = `0.0001` · LR B = `0.0004` · 3 epochs · seed 42


In [10]:
result_screen_4 = run_experiment(lr_A=0.0001, ratio=4, seed=42, stage='screening')


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,Accuracy,F1,Macro F1
200,0.585200,0.542960,0.705268,0.676037,0.702849
400,0.524500,0.478807,0.750581,0.708116,0.745188
600,0.479500,0.446501,0.780040,0.728566,0.771834
800,0.462000,0.424676,0.792110,0.737565,0.782724
1000,0.455300,0.427854,0.792654,0.752004,0.786929
1200,0.432700,0.405434,0.803562,0.730980,0.788140
1400,0.436700,0.396827,0.806851,0.757597,0.798534
1600,0.423700,0.391007,0.812936,0.753608,0.801423
1800,0.419400,0.405543,0.802498,0.767344,0.797884
2000,0.407800,0.409331,0.803389,0.710619,0.780868


Step,Training Loss,Validation Loss,Accuracy,F1,Macro F1
200,0.585200,0.542960,0.705268,0.676037,0.702849
400,0.524500,0.478807,0.750581,0.708116,0.745188
600,0.479500,0.446501,0.780040,0.728566,0.771834
800,0.462000,0.424676,0.792110,0.737565,0.782724
1000,0.455300,0.427854,0.792654,0.752004,0.786929
1200,0.432700,0.405434,0.803562,0.730980,0.788140
1400,0.436700,0.396827,0.806851,0.757597,0.798534
1600,0.423700,0.391007,0.812936,0.753608,0.801423
1800,0.419400,0.405543,0.802498,0.767344,0.797884
2000,0.407800,0.409331,0.803389,0.710619,0.780868


{
  "stage": "screening",
  "run_id": "screening_A1e-04_ratio4_seed42",
  "seed": 42,
  "lr_A": 0.0001,
  "lr_B": 0.0004,
  "ratio": 4,
  "method": "LoRA+",
  "train_samples": 50000,
  "validation_samples": 40430,
  "epochs": 3.0,
  "accuracy": 0.8301014098441751,
  "f1": 0.7792099257497348,
  "macro_f1": 0.8205683993258513,
  "eval_loss": 0.3670630156993866,
  "train_runtime_seconds": 4014.0798,
  "total_runtime_seconds": 4116.627614198998,
  "peak_gpu_memory_gb": 1.5978035926818848,
  "trainable_parameters": 1181184,
  "global_steps": 4689,
  "subset_sha256": "4b6d64cbab6576ed317169322bdaf13629f11daabb7697c0034ee02de37c1c01"
}


### Screening 5/9 — LoRA+ | λ = 8

LR A = `0.0001` · LR B = `0.0008` · 3 epochs · seed 42


In [8]:
result_screen_5 = run_experiment(lr_A=0.0001, ratio=8, seed=42, stage='screening')


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,Accuracy,F1,Macro F1
4200,0.343300,0.360688,0.839179,0.786904,0.828881
4400,0.336800,0.357897,0.839253,0.788973,0.829578
4600,0.338200,0.360820,0.837843,0.782828,0.826723


{
  "stage": "screening",
  "run_id": "screening_A1e-04_ratio8_seed42",
  "seed": 42,
  "lr_A": 0.0001,
  "lr_B": 0.0008,
  "ratio": 8,
  "method": "LoRA+",
  "train_samples": 50000,
  "validation_samples": 40430,
  "epochs": 3.0,
  "accuracy": 0.8380163245115013,
  "f1": 0.7835971318111224,
  "macro_f1": 0.8270812901777513,
  "eval_loss": 0.35979968309402466,
  "train_runtime_seconds": 539.5543,
  "total_runtime_seconds": 646.69943826,
  "peak_gpu_memory_gb": 1.5915770530700684,
  "trainable_parameters": 1181184,
  "global_steps": 4689,
  "subset_sha256": "4b6d64cbab6576ed317169322bdaf13629f11daabb7697c0034ee02de37c1c01"
}


### Screening 6/9 — LoRA+ | λ = 16

LR A = `0.0001` · LR B = `0.0016` · 3 epochs · seed 42


In [11]:
result_screen_6 = run_experiment(lr_A=0.0001, ratio=16, seed=42, stage='screening')


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,Accuracy,F1,Macro F1
1200,0.411600,0.387024,0.812862,0.727920,0.792653
1400,0.411700,0.371167,0.824512,0.778468,0.816589
1600,0.396200,0.386114,0.823423,0.754310,0.808250
1800,0.380400,0.387115,0.816720,0.782838,0.812147
2000,0.380800,0.373255,0.826268,0.757993,0.811245
2200,0.386600,0.379326,0.822508,0.745532,0.804631
2400,0.358800,0.358820,0.836903,0.791725,0.828850
2600,0.359500,0.358567,0.838115,0.778863,0.825594
2800,0.360100,0.354439,0.837769,0.790347,0.829021
3000,0.357500,0.350179,0.840267,0.793608,0.831664


{
  "stage": "screening",
  "run_id": "screening_A1e-04_ratio16_seed42",
  "seed": 42,
  "lr_A": 0.0001,
  "lr_B": 0.0016,
  "ratio": 16,
  "method": "LoRA+",
  "train_samples": 50000,
  "validation_samples": 40430,
  "epochs": 3.0,
  "accuracy": 0.8444471926787039,
  "f1": 0.7948458652748328,
  "macro_f1": 0.8347897287732595,
  "eval_loss": 0.3491663336753845,
  "train_runtime_seconds": 2911.3221,
  "total_runtime_seconds": 3018.2646376350003,
  "peak_gpu_memory_gb": 1.5969557762145996,
  "trainable_parameters": 1181184,
  "global_steps": 4689,
  "subset_sha256": "4b6d64cbab6576ed317169322bdaf13629f11daabb7697c0034ee02de37c1c01"
}


### Screening 7/9 — LoRA+ | λ = 4

LR A = `0.0002` · LR B = `0.0008` · 3 epochs · seed 42


In [ ]:
result_screen_7 = run_experiment(lr_A=0.0002, ratio=4, seed=42, stage='screening')


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,Accuracy,F1,Macro F1
200,0.539200,0.487417,0.751744,0.705789,0.745536
400,0.494700,0.448592,0.775068,0.731835,0.769066
600,0.458900,0.419978,0.794361,0.746432,0.786741
800,0.435600,0.405135,0.806505,0.749848,0.796042
1000,0.440800,0.401707,0.807841,0.761694,0.800354
1200,0.413400,0.401341,0.807865,0.715604,0.785266
1400,0.416700,0.380666,0.818105,0.772927,0.810608
1600,0.400200,0.379380,0.823126,0.769523,0.813012
1800,0.386800,0.398483,0.811007,0.778772,0.806907
2000,0.392800,0.374431,0.821370,0.751428,0.806012


### Screening 8/9 — LoRA+ | λ = 8

LR A = `0.0002` · LR B = `0.0016` · 3 epochs · seed 42


In [ ]:
result_screen_8 = run_experiment(lr_A=0.0002, ratio=8, seed=42, stage='screening')


### Screening 9/9 — LoRA+ | λ = 16

LR A = `0.0002` · LR B = `0.0032` · 3 epochs · seed 42


In [ ]:
result_screen_9 = run_experiment(lr_A=0.0002, ratio=16, seed=42, stage='screening')


## 8. Select winners using final validation **binary F1** (accuracy tie-breaker)

In [ ]:
def read_results(stage):
    files=sorted((ROOT/stage).glob('*/result.json'))
    return pd.DataFrame([json.loads(p.read_text()) for p in files]) if files else pd.DataFrame()

screen = read_results('screening')
if screen.empty:
    print('No completed screening runs yet. Run section 7 first.')
else:
    display(screen[['method','lr_A','lr_B','ratio','seed','accuracy','f1','train_runtime_seconds','peak_gpu_memory_gb']]
            .sort_values(['method','f1'],ascending=[True,False]))
    print('Completed:',len(screen),'of',len(SCREENING))
    if len(screen)<len(SCREENING): print('WARNING: Screening incomplete. Do not finalize winners yet.')
    if len(screen)==len(SCREENING):
        winners=(screen.sort_values(['f1','accuracy'],ascending=False)
                      .groupby('method',as_index=False).first())
        print('Selected winners:')
        display(winners[['method','lr_A','lr_B','ratio','accuracy','f1']])
        (ROOT/'selected_winners.json').write_text(winners[['method','lr_A','ratio']].to_json(orient='records'))

## 9. Final evaluation: 2 selected methods × 3 model seeds
Runs only if all nine screening experiments are complete and a selection file exists. **Each run starts from GPT-2 pretrained weights** and uses the same fixed 50k subset. Change `RUN_FINAL=True` when ready.

**Methodological caveat:** screening and final runs use the same official validation set, so the final validation accuracy is not an independent held-out test estimate. The report should disclose this; the official QQP test labels are unavailable.

In [ ]:
RUN_FINAL = False  # CHANGE TO True AFTER SCREENING IS COMPLETE
if RUN_FINAL:
    screen = read_results('screening')
    assert len(screen)==len(SCREENING), 'Finish all nine screening runs first.'
    selected_file=ROOT/'selected_winners.json'
    assert selected_file.exists(), 'Run section 8 to select winners.'
    winners=json.loads(selected_file.read_text())
    assert {x['method'] for x in winners}=={'LoRA','LoRA+'}
    for win in winners:
        for seed in CFG['final_seeds']:
            print('FINAL',win['method'],'seed',seed)
            run_experiment(lr_A=float(win['lr_A']),ratio=int(win['ratio']),
                           seed=seed,stage='final',save_adapter=True)

## 10. Results, statistical summary, and charts

In [ ]:
final=read_results('final')
if final.empty:
    print('No final results yet. Complete section 9 first.')
else:
    display(final[['method','seed','lr_A','lr_B','ratio','accuracy','f1',
                   'train_runtime_seconds','peak_gpu_memory_gb','trainable_parameters']]
            .sort_values(['method','seed']))
    summary=final.groupby('method').agg(
        n=('accuracy','size'), accuracy_mean=('accuracy','mean'),
        accuracy_std=('accuracy','std'), f1_mean=('f1','mean'),f1_std=('f1','std'),
        runtime_mean_sec=('train_runtime_seconds','mean'),
        peak_memory_mean_gb=('peak_gpu_memory_gb','mean'),
        trainable_parameters=('trainable_parameters','first')).reset_index()
    display(summary)
    summary.to_csv(ROOT/'final_summary.csv',index=False)
    final.to_csv(ROOT/'final_results.csv',index=False)
    if set(final.method)=={'LoRA','LoRA+'}:
        fig,ax=plt.subplots(figsize=(7,4))
        ax.bar(summary['method'],summary['accuracy_mean'],yerr=summary['accuracy_std'].fillna(0),capsize=5)
        ax.set_ylabel('QQP validation accuracy');ax.set_title('GPT-2 QQP: LoRA vs LoRA+ (mean ± SD)')
        fig.tight_layout();fig.savefig(ROOT/'accuracy_comparison.png',dpi=180);plt.show()

## 11. Plot training convergence (loss vs global step)

In [ ]:
import glob
logs=sorted((ROOT/'final').glob('*/trainer_log_history.json'))
if not logs:
    print('No final-run training logs yet.')
else:
    fig,ax=plt.subplots(figsize=(9,5))
    for p in logs:
        entries=json.loads(p.read_text())
        points=[x for x in entries if 'loss' in x and 'step' in x]
        if points:
            ax.plot([x['step'] for x in points],[x['loss'] for x in points],label=p.parent.name)
    ax.set(xlabel='Optimizer steps',ylabel='Training loss',title='GPT-2 QQP training convergence')
    ax.legend(fontsize=7);fig.tight_layout();fig.savefig(ROOT/'convergence.png',dpi=180);plt.show()

## 12. Reporting notes

This is the team’s reduced-compute protocol, not an exact reproduction of the paper. Selection and reporting reuse the official validation split. Screening seed 42 is also included in the final three-seed results. LoRA and LoRA+ should be compared on matched dataset, training budget and hardware. LR for the classification head is set to LR A as an implementation choice. Checkpoints save every 1000 optimizer steps and may not exist if Colab interrupts earlier. GPU runtime and availability vary.